In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ims

In [ ]:
ROOT = Path(r"C:\Users\user\PycharmProjects\PythonProject\data")
OUT  = Path(r"C:\Users\user\PycharmProjects\PythonProject\data\output_aug"); OUT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(7)

def add_gaussian_noise(arr, snr_db=18.0, clip_zero=True):
    arr = np.clip(arr, 0, None) if clip_zero else arr
    power = float(np.mean(arr**2)) if arr.size else 0.0
    snr = 10**(snr_db/10)
    sigma = np.sqrt(power / snr) if snr > 0 else 0.0
    out = arr + rng.normal(0.0, sigma, arr.shape)
    return np.clip(out, 0, None) if clip_zero else out

def add_poisson_noise(arr, scale=0.5, clip_zero=True):
    lam = np.maximum(arr * scale, 0.0)
    out = rng.poisson(lam).astype(np.float32) / max(scale, 1e-6)
    return np.clip(out, 0, None) if clip_zero else out

def multiplicative_jitter(arr, sigma=0.05, clip_zero=True):
    out = arr * rng.normal(1.0, sigma, size=arr.shape)
    return np.clip(out, 0, None) if clip_zero else out

def small_axis_shift(arr, rt_px=2, dt_px=2):
    out = arr
    if rt_px != 0:
        out = np.roll(out, rt_px, axis=0)
    if dt_px != 0:
        out = np.roll(out, dt_px, axis=1)
    return out

def process_mea(mea_path: Path, snr_db=18.0, jitter_sigma=0.05, rt_shift=(-2,2), dt_shift=(-2,2)):
    spec = ims.Spectrum.read_mea(str(mea_path))
    M = np.clip(spec.values.astype(np.float32), 0, None)
    rt_px = rng.integers(rt_shift[0], rt_shift[1]+1) if rt_shift[0] <= rt_shift[1] else 0
    dt_px = rng.integers(dt_shift[0], dt_shift[1]+1) if dt_shift[0] <= dt_shift[1] else 0
    A = small_axis_shift(M, rt_px=rt_px, dt_px=dt_px)
    A = multiplicative_jitter(A, sigma=jitter_sigma)
    A = add_gaussian_noise(A, snr_db=snr_db)

    base = mea_path.stem
    np.save(OUT / f"{base}_M.npy", M)
    np.save(OUT / f"{base}_M_aug.npy", A)
    np.save(OUT / f"{base}_rt.npy", spec.ret_time)
    np.save(OUT / f"{base}_dt.npy", spec.drift_time)

    return {
        "file": str(mea_path),
        "shape": f"{M.shape[0]}x{M.shape[1]}",
        "rt_shift_px": int(rt_px),
        "dt_shift_px": int(dt_px),
        "jitter_sigma": float(jitter_sigma),
        "snr_db": float(snr_db),
        "clean_min": float(M.min()),
        "clean_max": float(M.max()),
        "aug_min": float(A.min()),
        "aug_max": float(A.max()),
    }

files = sorted(ROOT.rglob("*.mea"))
log = []
for f in files:
    try:
        log.append(process_mea(f, snr_db=18.0, jitter_sigma=0.05, rt_shift=(-2,2), dt_shift=(-2,2)))
    except Exception as e:
        log.append({"file": str(f), "error": repr(e)})

pd.DataFrame(log).to_csv(OUT / "augmentation_log.csv", index=False)
print(f"Processed {len(files)} files → {OUT}")


In [ ]:
MEA_ROOT  = Path(r"C:\Users\user\Desktop\Thesis_project\IMS_Honey_Botanical")
DATA_ROOT = Path(r"C:\Users\user\Desktop\Thesis_project\IMS_Honey_Botanical\output_aug")
N = 20

pairs = []
for mea in sorted(MEA_ROOT.rglob("*.mea")):
    stem = mea.stem
    aug = DATA_ROOT / f"{stem}_M_aug.npy"
    if not aug.exists():
        alt = DATA_ROOT / f"{stem}_M_noisy.npy"
        if alt.exists():
            aug = alt
        else:
            continue
    pairs.append((mea, aug))
    if len(pairs) >= N:
        break

print(f"Comparing {len(pairs)} pairs")

def psnr(x, y):
    x = x.astype(np.float64); y = y.astype(np.float64)
    mse = np.mean((x - y) ** 2)
    if mse == 0:
        return float("inf"), 0.0
    peak = np.max(x) - np.min(x)
    return 20 * np.log10(peak / np.sqrt(mse + 1e-12)), mse

for mea_path, aug_path in pairs:
    spec = ims.Spectrum.read_mea(str(mea_path))
    clean = np.clip(spec.values.astype(np.float32), 0, None)
    aug = np.load(aug_path).astype(np.float32)
    vmin, vmax = float(np.quantile(clean, 0.01)), float(np.quantile(clean, 0.99))

    p, mse = psnr(clean, aug)
    print(f"{mea_path.name}  vs  {aug_path.name}  ->  PSNR: {p:.2f} dB, MSE: {mse:.4g}")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    im0 = axes[0].imshow(clean, aspect="auto", origin="lower", vmin=vmin, vmax=vmax, cmap="viridis")
    axes[0].set_title(f"Clean: {mea_path.name}")
    axes[0].set_xlabel("Drift time (index)")
    axes[0].set_ylabel("Retention time (index)")

    im1 = axes[1].imshow(aug, aspect="auto", origin="lower", vmin=vmin, vmax=vmax, cmap="viridis")
    axes[1].set_title(f"Augmented: {aug_path.name}")
    axes[1].set_xlabel("Drift time (index)")

    diff = aug - clean
    dlim = float(np.quantile(np.abs(diff), 0.995))
    im2 = axes[2].imshow(diff, aspect="auto", origin="lower", vmin=-dlim, vmax=dlim, cmap="seismic")
    axes[2].set_title("Augmented − Clean (diff)")
    axes[2].set_xlabel("Drift time (index)")

    cbar = fig.colorbar(im1, ax=axes[:2], shrink=0.8)
    cbar.set_label("Intensity")
    cbar2 = fig.colorbar(im2, ax=[axes[2]], shrink=0.8)
    cbar2.set_label("Δ Intensity")

    plt.show()


### Reliability Evaluation – Classification of Original vs Augmented Data
This section automatically scans `DATA_ROOT` for all matching `_M.npy` and `_M_aug.npy` files.
It loads and flattens them, combines all pairs, and runs three classifiers (Logistic Regression, Random Forest, SVM)
to evaluate whether augmented (noise-added) data can be distinguished from original data.

Metrics printed: Accuracy, Precision, Recall, F1, ROC-AUC.

> If classifiers perform well → augmented data differs noticeably (less reliable)
> If classifiers perform poorly (~0.5 ROC-AUC) → augmented data closely resembles real data (more reliable)

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def bin_array(arr, binning=2):
    if binning <= 1:
        return arr
    if arr.ndim == 2:
        r, c = arr.shape
        r2, c2 = r // binning, c // binning
        arr = arr[:r2 * binning, :c2 * binning]
        return arr.reshape(r2, binning, c2, binning).mean(axis=(1, 3))
    elif arr.ndim == 3:
        n, r, c = arr.shape
        r2, c2 = r // binning, c // binning
        arr = arr[:, :r2 * binning, :c2 * binning]
        return arr.reshape(n, r2, binning, c2, binning).mean(axis=(2, 4))
    return arr

BINNING = 4
MAX_SAMPLES_PER_FILE = 500

pairs = []
for fname in os.listdir(DATA_ROOT):
    if fname.endswith("_M.npy"):
        base = fname[:-6]
        aug_name = base + "_M_aug.npy"
        aug_path = os.path.join(DATA_ROOT, aug_name)
        orig_path = os.path.join(DATA_ROOT, fname)
        if os.path.exists(aug_path):
            pairs.append((orig_path, aug_path))

if not pairs:
    raise RuntimeError("No matching _M.npy and _M_aug.npy pairs found in DATA_ROOT")

X_parts, y_parts = [], []

for orig_path, aug_path in pairs:
    Xo = np.load(orig_path)
    Xa = np.load(aug_path)
    if Xo.shape != Xa.shape:
        print("shape mismatch, skipping:", orig_path)
        continue

    Xo = bin_array(Xo, BINNING)
    Xa = bin_array(Xa, BINNING)

    n_take = min(len(Xo), MAX_SAMPLES_PER_FILE)
    idx_o = np.random.choice(len(Xo), n_take, replace=False)
    idx_a = np.random.choice(len(Xa), n_take, replace=False)

    X_parts.append(Xo[idx_o].reshape(n_take, -1).astype(np.float32))
    y_parts.append(np.zeros(n_take, dtype=np.int8))
    X_parts.append(Xa[idx_a].reshape(n_take, -1).astype(np.float32))
    y_parts.append(np.ones(n_take, dtype=np.int8))

X = np.concatenate(X_parts, axis=0)
y = np.concatenate(y_parts, axis=0)

print(f"Loaded {len(X)} total samples ({np.sum(y==0)} original / {np.sum(y==1)} augmented)")
print("Feature dimension:", X.shape[1])

X = X.astype(np.float32)
perm = np.random.permutation(len(X))
X = X[perm]
y = y[perm]

MAX_SAMPLES_TOTAL = 20000
if len(X) > MAX_SAMPLES_TOTAL:
    idx = np.random.choice(len(X), MAX_SAMPLES_TOTAL, replace=False)
    X = X[idx]
    y = y[idx]
    print(f"Downsampled to {len(X)} samples total.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)
}

for name, clf in classifiers.items():
    clf.fit(X_train_s, y_train)
    y_pred = clf.predict(X_test_s)
    y_proba = clf.predict_proba(X_test_s)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall   : {rec:.3f}")
    print(f"F1-score : {f1:.3f}")
    print(f"ROC-AUC  : {auc:.3f}")

